In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime
import time
import random

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-IN,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Referer": "https://www.amazon.in/",
    "DNT": "1"
}

In [3]:
def get_page(url):
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            return BeautifulSoup(response.content, "html.parser")
        else:
            print(f"Got status {response.status_code} for {url}")
            return None
    except Exception as e:
        print(f"Error fetching page: {e}")
        return None

In [4]:
def parse_products(soup):
    products = []
    
    results = soup.find_all("div", {"data-component-type": "s-search-result"})
    
    for item in results:
        try:
            title_tag = item.find("span", {"class": "a-size-medium"})
            if not title_tag:
                title_tag = item.find("span", {"class": "a-size-base-plus"})
            title = title_tag.get_text(strip=True) if title_tag else "N/A"

            img_tag = item.find("img", {"class": "s-image"})
            image = img_tag["src"] if img_tag else "N/A"

            rating_tag = item.find("span", {"class": "a-icon-alt"})
            rating = rating_tag.get_text(strip=True) if rating_tag else "N/A"

            price_whole = item.find("span", {"class": "a-price-whole"})
            price = price_whole.get_text(strip=True).replace(",", "") if price_whole else "N/A"
            if price != "N/A":
                price = "₹" + price

            sponsored_tag = item.find("span", string=lambda t: t and "Sponsored" in t)
            result_type = "Ad" if sponsored_tag else "Organic"

            products.append({
                "Title": title,
                "Image URL": image,
                "Rating": rating,
                "Price": price,
                "Result Type": result_type
            })

        except Exception as e:
            print(f"Skipping one product due to error: {e}")
            continue
    
    return products

In [5]:
def get_next_page_url(soup):
    next_btn = soup.find("a", {"class": "s-pagination-next"})
    if next_btn and next_btn.get("href"):
        return "https://www.amazon.in" + next_btn["href"]
    return None

In [6]:
base_url = "https://www.amazon.in/s?k=laptops&rh=n%3A1375424031&ref=nb_sb_noss"

all_products = []
max_pages = 5
current_url = base_url
page_num = 1

while current_url and page_num <= max_pages:
    print(f"Scraping page {page_num}...")
    soup = get_page(current_url)
    
    if soup is None:
        print("Could not fetch page, stopping.")
        break
    
    page_products = parse_products(soup)
    all_products.extend(page_products)
    print(f"Found {len(page_products)} products on page {page_num}")
    
    current_url = get_next_page_url(soup)
    page_num += 1
    time.sleep(random.uniform(2, 4))

print(f"\nTotal products scraped: {len(all_products)}")

Scraping page 1...
Found 30 products on page 1
Scraping page 2...
Found 30 products on page 2
Scraping page 3...
Found 30 products on page 3
Scraping page 4...
Found 30 products on page 4
Scraping page 5...
Found 30 products on page 5

Total products scraped: 150


In [7]:
df = pd.DataFrame(all_products)
df.drop_duplicates(subset=["Title"], inplace=True)
df.reset_index(drop=True, inplace=True)
print(df.shape)
df.head(10)

(1, 5)


,Title,Image URL,Rating,Price,Result Type
0,N/A,https://m.media-amazon.com/images/I/713cu-sW3T...,4.4 out of 5 stars,₹19490,Ad


In [8]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"amazon_laptops_{timestamp}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"Data saved to {filename}")

Data saved to amazon_laptops_20260518_163711.csv
